# Hamrobazaar Car/Bike Scraper
This notebook demonstrates how to scrape car and bike listings from the Hamrobazaar API, fetch detailed product attributes, and save the results to a CSV file. The workflow includes API setup, parallel data fetching, and CSV export.

## 1. Import Required Libraries
We use requests for HTTP calls, csv for file writing, and ThreadPoolExecutor for parallel processing.

In [1]:
import requests
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed

## 2. Output File Setup
Specify the output CSV file where scraped data will be saved.

In [2]:
output_file = "test100.csv"

## 3. API Headers
Set up the required headers for Hamrobazaar API requests. These may change in the future if the API is updated.

In [3]:
HEADERS = {
    "accept": "application/json, text/plain, */*",
    "accept-language": "en-US,en;q=0.9",
    "access-control-allow-origin": "*",
    "apikey": "09BECB8F84BCB7A1796AB12B98C1FB9E",
    "cache-control": "no-cache",
    "country_code": "null",
    "deviceid": "2d426b8a-b97e-4b90-ad4e-f9244a18ed43",
    "devicesource": "mobile",
    "pragma": "no-cache",
    "sec-ch-ua": '"Chromium";v="142", "Microsoft Edge";v="142", "Not_A Brand";v="99"',
    "sec-ch-ua-mobile": "?1",
    "sec-ch-ua-platform": '"Android"',
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-site",
    "strict-transport-security": "max-age=2592000",
    "x-content-type-options": "nosniff",
    "x-frame-options": "SAMEORIGIN",
    "Referer": "https://hamrobazaar.com/",
}

## 4. Category IDs
Define the category IDs for cars and bikes. These are used to filter listings by type.

In [4]:
# For car the url is https://api.hamrobazaar.com/api/Product?PageSize=27&CategoryId=F93D355F-CC20-4FFE-9CB7-6C7CDFF1DC50&IsHBSelect=false&PageNumber=2
# For bikes the url is https://api.hamrobazaar.com/api/Product?PageSize=27&CategoryId=59973AED-F03D-4985-9AEC-542831929081&IsHBSelect=false&PageNumber=3

car_category_id = "F93D355F-CC20-4FFE-9CB7-6C7CDFF1DC50"
bike_category_id = "59973AED-F03D-4985-9AEC-542831929081"

## 5. Fetching Product Details
- Define a function to fetch detailed attributes for each product entry using its unique ID.
- Creating a dictionary mapping attribute names to values. 

In [ ]:
def get_entry_details(id: str) -> dict:
    url = f"https://api.hamrobazaar.com/api/Product/{id}"
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    data = response.json()
    attributes = data.get("data", {}).get("productAttributeValues", [])
    
    # Dictionary mapping attribute names to values
    details_dict = {}
    for attr in attributes:
        attr_name = attr.get("attributeName", "")
        # Removing , in the value to avoid csv escape.
        attr_value = attr.get("value", "").replace(",", " ")
        details_dict[attr_name] = attr_value
    
    return details_dict

## 6. Fetching Pages of Listings
- Define a function to fetch a page of product listings and retrieve details for each entry.
- Mapping each header to its corresponding value. 

In [6]:
# enter how many pages you need here. Each page has roughly 
page_number = 10 

In [7]:
def fetch_page(page_number: int,category_id: str) -> list:
    url = (
        f"https://api.hamrobazaar.com/api/Product?PageSize=27&CategoryId={category_id}&IsHBSelect=false&PageNumber={page_number}"
    )
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    data = response.json()
    entries = data.get("data", [])
    rows = []
    
    for entry in entries:
        entry_id = entry.get("id", "").replace("-", "")
        details = get_entry_details(entry_id)
        entry_name = entry.get("name", "").replace(",", " ")
        entry_price = entry.get("price", "")
        
        # Mapping each header to its corresponding value and using empty string if not found anything.
        row = [
            entry_name,
            entry_price,
            details.get("Used For", ""),
            details.get("Warranty", ""),
            details.get("Transmission", ""),
            details.get("Colour", ""),
            details.get("Make Year", ""),
            details.get("Features", ""),
            details.get("Mileage", ""),
            details.get("Engine (CC)", ""),
            details.get("Fuel", ""),
            details.get("Kilometer Run", ""),
            details.get("Types", ""),
        ]
        rows.append(row)
    
    return rows

## 7. CSV Header Setup
Specify the column names for the output CSV file.

In [8]:
header = [
    "Name",
    "Price",
    "Used For",
    "Warranty",
    "Transmission",
    "Colour",
    "Make Year",
    "Features",
    "Mileage",
    "Engine (CC)",
    "Fuel",
    "Kilometer Run",
    "Types",
]


## 8. Page Range Setup
Set the range of pages to fetch and initialize the list to store all rows.

In [9]:
page_numbers = list(range(1, page_number+1))
all_rows = []

## 9. Parallel Fetching
Use ThreadPoolExecutor to fetch multiple pages in parallel, speeding up the scraping process.

In [10]:
print(f"Fetching pages 1 to {page_number} in parallel...")

Fetching pages 1 to 10 in parallel...


In [11]:
# specify the maximum number of workers you want to use in parallel
max_workers = 10 

Using ThreadPoolExecutor

In [12]:
with ThreadPoolExecutor(max_workers) as executor:
    # Using car_category_id here. the category id needs to be changed here. 
    futures = {executor.submit(fetch_page, page,car_category_id): page for page in page_numbers}
    for future in as_completed(futures):
        page = futures[future]
        try:
            rows = future.result()
            all_rows.extend(rows)
            print(f"Fetched page {page}")
        except Exception as e:
            print(f"Error fetching page {page}: {e}")

Fetched page 2
Fetched page 1
Fetched page 4
Fetched page 6
Fetched page 3
Fetched page 5
Fetched page 10
Fetched page 9
Fetched page 8
Error fetching page 7: HTTPSConnectionPool(host='api.hamrobazaar.com', port=443): Max retries exceeded with url: /api/Product/42CF27F2128B1ABC9CA2579C765F27E2 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000002031B212930>, 'Connection to api.hamrobazaar.com timed out. (connect timeout=None)'))


## 10. Writing to csv


In [13]:
with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(all_rows)

print(f"CSV file written: {output_file}")

CSV file written: test100.csv
